# Task 4 — Round 1 model search

Original and engineered versions of each classifier are in separate cells, so a candidate can be edited, added, or removed independently. All candidates use the same stratified hash-group-aware 80/20 validation split and save their own model/evaluation cards. Existing results are read from saved artifacts; names are never silently overwritten.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
RAW = SRC / "task4" / "data" / "raw"
from src.pipeline.random_seed import set_seed
SEED = set_seed()
import json
import numpy as np
import pandas as pd
from IPython.display import display

import joblib
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.pipeline.read_evalutate_summarize import read_evaluation_index

# Load saved EDA files through their JSON cards

In [2]:
def load_data(name):
    card = json.loads((SRC / "task4" / "data" / "eda" / f"{name}.json").read_text(encoding="utf-8"))
    read = pd.read_parquet if card["format"] == "parquet" else pd.read_csv
    train = read(SRC / card["splits"]["train"]["path"])
    test = read(SRC / card["splits"]["test"]["path"])
    assert train.id.equals(pd.read_csv(RAW / "train.csv").id)
    assert test.id.equals(pd.read_csv(RAW / "test.csv").id)
    return train, test

original_train, original_test = load_data("eda_original_v1")
engineered_train, engineered_test = load_data("eda_1_v1")
assert original_train["class"].equals(engineered_train["class"])
assert original_train["image_sha256"].equals(engineered_train["image_sha256"])
exclude = {"id", "class", "image_sha256", "width", "height"}
features = {"original": [c for c in original_test if c not in exclude],
            "engineered": [c for c in engineered_test if c not in exclude]}
datasets = {"original": original_train, "engineered": engineered_train}
labels = sorted(original_train["class"].unique())
print("Feature counts:", {k: len(v) for k, v in features.items()})

Feature counts: {'original': 384, 'engineered': 1228}


# Shared validation split

In [3]:
y = original_train["class"]
groups = original_train["image_sha256"]
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_idx, valid_idx = next(splitter.split(original_train, y, groups))
assert not (set(groups.iloc[dev_idx]) & set(groups.iloc[valid_idx]))
print("Development/validation:", len(dev_idx), len(valid_idx))
display(pd.DataFrame({"development": y.iloc[dev_idx].value_counts(),
                      "validation": y.iloc[valid_idx].value_counts()}))

Development/validation: 25236 6310


,development,validation
class,,
7-11k,9943,2486
pube,4741,1185
freefried,4477,1119
fiveN,3525,882
roblock,2550,638


# Fit, save, and evaluate candidate classifiers

In [4]:
DATA_NAMES = {"original": "eda_original_v1", "engineered": "eda_1_v1"}
X_dev = {key: frame.loc[dev_idx, features[key]].astype("float32") for key, frame in datasets.items()}
X_valid = {key: frame.loc[valid_idx, features[key]].astype("float32") for key, frame in datasets.items()}
y_dev, y_valid = y.iloc[dev_idx], y.iloc[valid_idx]
SAVE_DIR = SRC / "task4" / "save"

def saved_result(name, dataset):
    model_card_path = SAVE_DIR / f"{name}.json"
    eval_path = SAVE_DIR / f"{name}-evaluation.json"
    if not model_card_path.exists() and not eval_path.exists():
        return None
    if model_card_path.exists() != eval_path.exists():
        raise RuntimeError(f"Incomplete artifact for {name}; inspect it before rerunning")
    model_card = json.loads(model_card_path.read_text(encoding="utf-8"))
    if model_card["data_name"] != DATA_NAMES[dataset]:
        raise ValueError(f"{name} is linked to the wrong EDA dataset")
    result = json.loads(eval_path.read_text(encoding="utf-8"))
    print("Saved:", name, "macro F1", round(result["score"], 5))
    return result

def record_model(name, dataset, model, idea):
    pred = model.predict(X_valid[dataset])
    save_model(task=4, name=name, model=model, data_name=DATA_NAMES[dataset],
               card={"idea": idea, "features": dataset, "seed": SEED,
                     "fit_rows": len(dev_idx), "validation": "stratified hash-group fold 1/5"})
    result = evaluate_model(task=4, name=name, y_true=y_valid, y_pred=pred,
                            card={"split": "same stratified hash-group fold 1/5", "seed": SEED})
    print(name, "macro F1", round(result["score"], 5))
    return result

## Majority baseline

In [5]:
name = "t4_dummy_pixels_r1"
if saved_result(name, "original") is None:
    model = DummyClassifier(strategy="most_frequent")
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Majority-class reference")

Saved: t4_dummy_pixels_r1 macro F1 0.11305


## Logistic regression — original, then engineered

In [6]:
name = "t4_logreg_pixels_r1"
if saved_result(name, "original") is None:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=500, random_state=SEED))
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Linear pixel classifier")

Saved: t4_logreg_pixels_r1 macro F1 0.41192


In [7]:
name = "t4_logreg_features_r1"
if saved_result(name, "engineered") is None:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=600, random_state=SEED))
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Regularized linear engineered features")

Saved: t4_logreg_features_r1 macro F1 0.72128


## Linear SVM via SGD — original, then engineered

In [8]:
name = "t4_svm_pixels_r1"
if saved_result(name, "original") is None:
    model = make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", alpha=0.0001, max_iter=80, tol=0.001, random_state=SEED))
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Linear margin classifier on pixels")

Saved: t4_svm_pixels_r1 macro F1 0.33276


In [9]:
name = "t4_svm_features_r1"
if saved_result(name, "engineered") is None:
    model = make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", alpha=0.0001, max_iter=80, tol=0.001, random_state=SEED))
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Linear margin classifier on engineered features")

Saved: t4_svm_features_r1 macro F1 0.67859


## ExtraTrees — original, then engineered

In [10]:
name = "t4_extratrees_pixels_r1"
if saved_result(name, "original") is None:
    model = ExtraTreesClassifier(n_estimators=80, min_samples_leaf=2, max_features=0.15, max_depth=20, n_jobs=4, random_state=SEED)
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Randomized trees on pixels")

Saved: t4_extratrees_pixels_r1 macro F1 0.5257


In [11]:
name = "t4_extratrees_features_r1"
if saved_result(name, "engineered") is None:
    model = ExtraTreesClassifier(n_estimators=80, min_samples_leaf=2, max_features=0.15, max_depth=20, n_jobs=4, random_state=SEED)
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Randomized trees on engineered features")

Saved: t4_extratrees_features_r1 macro F1 0.61219


## Histogram gradient boosting — original, then engineered

In [12]:
name = "t4_hgb_pixels_r1"
if saved_result(name, "original") is None:
    model = HistGradientBoostingClassifier(max_iter=60, max_leaf_nodes=15, learning_rate=0.08, random_state=SEED)
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Histogram gradient boosting on pixels")

Saved: t4_hgb_pixels_r1 macro F1 0.52485


In [13]:
name = "t4_hgb_features_r1"
if saved_result(name, "engineered") is None:
    model = HistGradientBoostingClassifier(max_iter=60, max_leaf_nodes=15, learning_rate=0.08, random_state=SEED)
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Histogram gradient boosting on engineered features")

Saved: t4_hgb_features_r1 macro F1 0.71047


## LightGBM, 15 leaves — original, then engineered

In [14]:
name = "t4_lgbm_15_pixels_r1"
if saved_result(name, "original") is None:
    model = LGBMClassifier(n_estimators=140, num_leaves=15, learning_rate=0.05, max_depth=-1, verbosity=-1, n_jobs=4, random_state=SEED)
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Small-leaf LightGBM on pixels")

Saved: t4_lgbm_15_pixels_r1 macro F1 0.53547


In [15]:
name = "t4_lgbm_15_features_r1"
if saved_result(name, "engineered") is None:
    model = LGBMClassifier(n_estimators=140, num_leaves=15, learning_rate=0.05, max_depth=-1, verbosity=-1, n_jobs=4, random_state=SEED)
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Small-leaf LightGBM on engineered features")

Saved: t4_lgbm_15_features_r1 macro F1 0.7292


## LightGBM, 31 leaves — original, then engineered

In [16]:
name = "t4_lgbm_31_pixels_r1"
if saved_result(name, "original") is None:
    model = LGBMClassifier(n_estimators=140, num_leaves=31, learning_rate=0.05, max_depth=-1, verbosity=-1, n_jobs=4, random_state=SEED)
    model.fit(X_dev["original"], y_dev)
    record_model(name, "original", model, "Medium-leaf LightGBM on pixels")

Saved: t4_lgbm_31_pixels_r1 macro F1 0.5893


In [17]:
name = "t4_lgbm_31_features_r1"
if saved_result(name, "engineered") is None:
    model = LGBMClassifier(n_estimators=140, num_leaves=31, learning_rate=0.05, max_depth=-1, verbosity=-1, n_jobs=4, random_state=SEED)
    model.fit(X_dev["engineered"], y_dev)
    record_model(name, "engineered", model, "Medium-leaf LightGBM on engineered features")

Saved: t4_lgbm_31_features_r1 macro F1 0.76267


# Tune LightGBM within Round 1

Search the same four configurations on each feature set using three hash-group-aware development folds. Select by mean macro F1, then evaluate the refitted model once on the existing validation fold. Fold scores show sensitivity to the split; this is still an exploratory round because earlier baselines used that validation fold.

In [18]:
PARAM_GRID = {"num_leaves": [15, 31], "min_child_samples": [20, 60]}
CV_SPLITS = 3
CV_SEED = SEED + 101

def tune_lgbm(dataset, name):
    prior = saved_result(name, dataset)
    if prior is not None:
        card = json.loads((SAVE_DIR / f"{name}.json").read_text(encoding="utf-8"))["card"]
        display(pd.DataFrame(card["cv_results"]).sort_values("mean_macro_F1", ascending=False).round(5))
        print("Selected:", card["best_params"], "validation macro F1:", round(prior["score"], 5))
        return prior

    cv = StratifiedGroupKFold(n_splits=CV_SPLITS, shuffle=True, random_state=CV_SEED)
    search = GridSearchCV(
        LGBMClassifier(n_estimators=140, learning_rate=0.05, verbosity=-1,
                       n_jobs=4, random_state=SEED),
        param_grid=PARAM_GRID, scoring="f1_macro", cv=cv, refit=True,
        n_jobs=1, error_score="raise", return_train_score=False,
    )
    search.fit(X_dev[dataset], y_dev, groups=groups.iloc[dev_idx])
    summary = []
    for i, params in enumerate(search.cv_results_["params"]):
        row = {**params, "mean_macro_F1": float(search.cv_results_["mean_test_score"][i]),
               "std_macro_F1": float(search.cv_results_["std_test_score"][i])}
        row.update({f"fold_{j+1}_macro_F1": float(search.cv_results_[f"split{j}_test_score"][i])
                    for j in range(CV_SPLITS)})
        summary.append(row)
    display(pd.DataFrame(summary).sort_values("mean_macro_F1", ascending=False).round(5))
    prediction = search.best_estimator_.predict(X_valid[dataset])
    save_model(task=4, name=name, model=search.best_estimator_, data_name=DATA_NAMES[dataset],
               card={"idea": "Round 1 grouped-CV LightGBM grid search", "features": dataset,
                     "seed": SEED, "fit_rows": len(dev_idx), "cv": "3-fold stratified image-hash groups on development only",
                     "cv_seed": CV_SEED, "best_params": search.best_params_, "cv_results": summary})
    result = evaluate_model(task=4, name=name, y_true=y_valid, y_pred=prediction,
                            card={"split": "held-out fold 1/5 after development-only CV", "seed": SEED})
    print("Selected:", search.best_params_, "validation macro F1:", round(result["score"], 5))
    return result


In [19]:
tuned_original = tune_lgbm("original", "t4_lgbm_tuned_pixels_r1")


Saved: t4_lgbm_tuned_pixels_r1 macro F1 0.59073


,min_child_samples,num_leaves,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
3,60,31,0.57794,0.00393,0.57777,0.58284,0.57322
1,20,31,0.57472,0.00345,0.57668,0.57761,0.56988
0,20,15,0.53443,0.00884,0.53921,0.54205,0.52204
2,60,15,0.53327,0.00566,0.53846,0.53595,0.52540


Selected: {'min_child_samples': 60, 'num_leaves': 31} validation macro F1: 0.59073


In [20]:
tuned_engineered = tune_lgbm("engineered", "t4_lgbm_tuned_features_r1")


Saved: t4_lgbm_tuned_features_r1 macro F1 0.76267


,min_child_samples,num_leaves,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
1,20,31,0.75737,0.00244,0.75752,0.76028,0.75430
3,60,31,0.75712,0.00564,0.75621,0.76443,0.75071
2,60,15,0.73858,0.00133,0.73917,0.73983,0.73674
0,20,15,0.73702,0.00349,0.73392,0.74190,0.73524


Selected: {'min_child_samples': 20, 'num_leaves': 31} validation macro F1: 0.76267


# Tune every Round 1 family

Use the same 9,000 development images and three hash-group-aware folds for each search. Search scores are directly comparable within this section. Refit each selected configuration on all development images, then evaluate on the common validation fold. The majority classifier remains a control; its strategy search is not a substantive model improvement.

In [21]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.base import clone

SEARCH_SEED = SEED + 202
SEARCH_ROWS = 9000
SEARCH_FOLDS = 3
search_idx, _ = train_test_split(
    np.arange(len(dev_idx)), train_size=SEARCH_ROWS, stratify=y_dev, random_state=SEARCH_SEED
)
search_y = y_dev.iloc[search_idx]
search_groups = groups.iloc[dev_idx].iloc[search_idx]
assert len(search_idx) == SEARCH_ROWS and search_groups.nunique() == SEARCH_ROWS
print("Common tuning sample:", SEARCH_ROWS, "images; class counts:")
display(search_y.value_counts().to_frame("search_count"))

TUNE_SPECS = {
    "dummy": (
        lambda: DummyClassifier(random_state=SEED),
        {"strategy": ["most_frequent", "prior", "stratified"]}, 3,
    ),
    "logreg": (
        lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=600, random_state=SEED)),
        {"logisticregression__C": [0.1, 0.3, 1.0, 3.0],
         "logisticregression__class_weight": [None, "balanced"]}, 3,
    ),
    "svm": (
        lambda: make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", max_iter=100, tol=0.001, random_state=SEED)),
        {"sgdclassifier__alpha": [1e-5, 1e-4, 1e-3],
         "sgdclassifier__class_weight": [None, "balanced"],
         "sgdclassifier__penalty": ["l2", "elasticnet"]}, 3,
    ),
    "extratrees": (
        lambda: ExtraTreesClassifier(n_jobs=4, random_state=SEED),
        {"n_estimators": [80, 160], "max_depth": [15, 25],
         "min_samples_leaf": [1, 2, 4], "max_features": [0.1, 0.2]}, 3,
    ),
    "hgb": (
        lambda: HistGradientBoostingClassifier(random_state=SEED),
        {"max_iter": [60, 100], "max_leaf_nodes": [15, 31],
         "learning_rate": [0.05, 0.1], "min_samples_leaf": [20, 50]}, 3,
    ),
    "lgbm_15": (
        lambda: LGBMClassifier(num_leaves=15, verbosity=-1, n_jobs=4, random_state=SEED),
        {"n_estimators": [100, 180], "learning_rate": [0.03, 0.08],
         "min_child_samples": [20, 60]}, 3,
    ),
    "lgbm_31": (
        lambda: LGBMClassifier(num_leaves=31, verbosity=-1, n_jobs=4, random_state=SEED),
        {"n_estimators": [100, 180], "learning_rate": [0.03, 0.08],
         "min_child_samples": [20, 60]}, 3,
    ),
}

def tune_family(family, dataset):
    name = f"t4_{family}_tuned_{'pixels' if dataset == 'original' else 'features'}_r1"
    previous = saved_result(name, dataset)
    if previous is not None:
        saved_card = json.loads((SAVE_DIR / f"{name}.json").read_text(encoding="utf-8"))["card"]
        display(pd.DataFrame(saved_card["cv_results"]).sort_values("mean_macro_F1", ascending=False).round(5))
        print("Selected:", saved_card["best_params"], "validation macro F1:", round(previous["score"], 5))
        return previous

    factory, space, trials = TUNE_SPECS[family]
    cv = StratifiedGroupKFold(n_splits=SEARCH_FOLDS, shuffle=True, random_state=SEARCH_SEED)
    search = RandomizedSearchCV(
        factory(), param_distributions=space, n_iter=trials, scoring="f1_macro",
        cv=cv, refit=False, random_state=SEARCH_SEED, n_jobs=1, error_score="raise",
    )
    search.fit(X_dev[dataset].iloc[search_idx], search_y, groups=search_groups)
    results = []
    for i, params in enumerate(search.cv_results_["params"]):
        row = {"params": params,
               "mean_macro_F1": float(search.cv_results_["mean_test_score"][i]),
               "std_macro_F1": float(search.cv_results_["std_test_score"][i])}
        row.update({f"fold_{j+1}_macro_F1": float(search.cv_results_[f"split{j}_test_score"][i])
                    for j in range(SEARCH_FOLDS)})
        results.append(row)
    display(pd.DataFrame(results).sort_values("mean_macro_F1", ascending=False).round(5))
    best = clone(factory()).set_params(**search.best_params_)
    best.fit(X_dev[dataset], y_dev)
    prediction = best.predict(X_valid[dataset])
    save_model(task=4, name=name, model=best, data_name=DATA_NAMES[dataset],
               card={"idea": f"Round 1 randomized search: {family}", "features": dataset,
                     "seed": SEED, "search_seed": SEARCH_SEED, "search_rows": SEARCH_ROWS,
                     "fit_rows": len(dev_idx), "cv": "3-fold stratified image-hash groups on common development sample",
                     "best_params": search.best_params_, "cv_results": results})
    evaluation = evaluate_model(task=4, name=name, y_true=y_valid, y_pred=prediction,
                                card={"split": "held-out fold 1/5 after development-only random search",
                                      "seed": SEED, "search_rows": SEARCH_ROWS})
    print("Selected:", search.best_params_, "validation macro F1:", round(evaluation["score"], 5))
    return evaluation


Common tuning sample: 9000 images; class counts:


,search_count
class,
7-11k,3546
pube,1691
freefried,1597
fiveN,1257
roblock,909


## Dummy search

In [22]:
tune_family("dummy", "original")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
2,{'strategy': 'stratified'},0.20946,0.00971,0.19573,0.21676,0.21588
0,{'strategy': 'most_frequent'},0.11306,0.00002,0.11303,0.11306,0.11308
1,{'strategy': 'prior'},0.11306,0.00002,0.11303,0.11306,0.11308


Selected: {'strategy': 'stratified'} validation macro F1: 0.20404


{'name': 't4_dummy_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:22:28+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.20403838560419021,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_dummy_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_dummy_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

## Logreg search

In [23]:
tune_family("logreg", "original")


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
2,{'logisticregression__class_weight': 'balanced...,0.39297,0.00337,0.38910,0.39731,0.39250
1,"{'logisticregression__class_weight': None, 'lo...",0.37653,0.00171,0.37414,0.37742,0.37803
0,"{'logisticregression__class_weight': None, 'lo...",0.37310,0.00032,0.37276,0.37301,0.37354


Selected: {'logisticregression__class_weight': 'balanced', 'logisticregression__C': 0.1} validation macro F1: 0.4329


{'name': 't4_logreg_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:23:12+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.4329011448700575,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_logreg_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_logreg_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [24]:
tune_family("logreg", "engineered")


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 600 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=600).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
2,{'logisticregression__class_weight': 'balanced...,0.64594,0.00757,0.64416,0.63768,0.65597
1,"{'logisticregression__class_weight': None, 'lo...",0.61622,0.00556,0.60851,0.62141,0.61872
0,"{'logisticregression__class_weight': None, 'lo...",0.60253,0.00404,0.60318,0.59728,0.60711


Selected: {'logisticregression__class_weight': 'balanced', 'logisticregression__C': 0.1} validation macro F1: 0.71457


{'name': 't4_logreg_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:25:13+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.7145699189950033,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_logreg_tuned_features_r1.json',
 'model_path': 'task4/save/t4_logreg_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

## Svm search

In [25]:
tune_family("svm", "original")


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


... (3 outputs left out)

,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
2,"{'sgdclassifier__penalty': 'l2', 'sgdclassifie...",0.32079,0.01368,0.32287,0.33641,0.30310
1,"{'sgdclassifier__penalty': 'l2', 'sgdclassifie...",0.31143,0.02220,0.29712,0.34279,0.29438
0,"{'sgdclassifier__penalty': 'elasticnet', 'sgdc...",0.30306,0.01066,0.29994,0.31739,0.29185


Selected: {'sgdclassifier__penalty': 'l2', 'sgdclassifier__class_weight': None, 'sgdclassifier__alpha': 0.0001} validation macro F1: 0.35987


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


{'name': 't4_svm_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:25:56+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.35987354378762276,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_svm_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_svm_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [26]:
tune_family("svm", "engineered")


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


... (3 outputs left out)

,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
2,"{'sgdclassifier__penalty': 'l2', 'sgdclassifie...",0.62685,0.00221,0.62375,0.62801,0.62878
0,"{'sgdclassifier__penalty': 'elasticnet', 'sgdc...",0.62375,0.00050,0.62443,0.62361,0.62322
1,"{'sgdclassifier__penalty': 'l2', 'sgdclassifie...",0.61694,0.00179,0.61447,0.61866,0.61768


Selected: {'sgdclassifier__penalty': 'l2', 'sgdclassifier__class_weight': None, 'sgdclassifier__alpha': 0.0001} validation macro F1: 0.67655


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


{'name': 't4_svm_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:27:47+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.6765507561452097,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_svm_tuned_features_r1.json',
 'model_path': 'task4/save/t4_svm_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

## Extratrees search

In [27]:
tune_family("extratrees", "original")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 160, 'min_samples_leaf': 4, '...",0.41183,0.00237,0.40876,0.41451,0.41223
2,"{'n_estimators': 80, 'min_samples_leaf': 4, 'm...",0.40003,0.00461,0.39367,0.40194,0.40447
1,"{'n_estimators': 80, 'min_samples_leaf': 4, 'm...",0.38913,0.00761,0.37927,0.39780,0.39034


Selected: {'n_estimators': 160, 'min_samples_leaf': 4, 'max_features': 0.2, 'max_depth': 15} validation macro F1: 0.49213


{'name': 't4_extratrees_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:28:14+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.4921318108034442,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_extratrees_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_extratrees_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [28]:
tune_family("extratrees", "engineered")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 160, 'min_samples_leaf': 4, '...",0.51385,0.00054,0.51427,0.51419,0.51308
2,"{'n_estimators': 80, 'min_samples_leaf': 4, 'm...",0.48391,0.00410,0.48849,0.48470,0.47855
1,"{'n_estimators': 80, 'min_samples_leaf': 4, 'm...",0.47715,0.00476,0.48205,0.47868,0.47071


Selected: {'n_estimators': 160, 'min_samples_leaf': 4, 'max_features': 0.2, 'max_depth': 15} validation macro F1: 0.59999


{'name': 't4_extratrees_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:29:56+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.5999914165261127,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_extratrees_tuned_features_r1.json',
 'model_path': 'task4/save/t4_extratrees_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

## Hgb search

In [29]:
tune_family("hgb", "original")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'min_samples_leaf': 50, 'max_leaf_nodes': 31,...",0.50368,0.00449,0.50077,0.50025,0.51003
1,"{'min_samples_leaf': 20, 'max_leaf_nodes': 15,...",0.47660,0.00367,0.47158,0.48026,0.47795
2,"{'min_samples_leaf': 50, 'max_leaf_nodes': 15,...",0.44638,0.00385,0.44417,0.45180,0.44317


Selected: {'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 60, 'learning_rate': 0.1} validation macro F1: 0.57489


{'name': 't4_hgb_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:31:16+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.5748852994618694,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_hgb_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_hgb_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [30]:
tune_family("hgb", "engineered")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'min_samples_leaf': 50, 'max_leaf_nodes': 31,...",0.69177,0.00480,0.68574,0.69208,0.69747
1,"{'min_samples_leaf': 20, 'max_leaf_nodes': 15,...",0.68432,0.00601,0.67591,0.68749,0.68956
2,"{'min_samples_leaf': 50, 'max_leaf_nodes': 15,...",0.64622,0.00547,0.63887,0.64781,0.65198


Selected: {'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 60, 'learning_rate': 0.1} validation macro F1: 0.75287


{'name': 't4_hgb_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:35:16+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.752871040552886,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_hgb_tuned_features_r1.json',
 'model_path': 'task4/save/t4_hgb_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

## Lgbm 15 search

In [31]:
tune_family("lgbm_15", "original")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 100, 'min_child_samples': 60,...",0.49129,0.00301,0.48934,0.49554,0.48900
1,"{'n_estimators': 100, 'min_child_samples': 20,...",0.48548,0.00880,0.47841,0.49789,0.48014
2,"{'n_estimators': 180, 'min_child_samples': 20,...",0.47443,0.00608,0.47829,0.47917,0.46585


Selected: {'n_estimators': 100, 'min_child_samples': 60, 'learning_rate': 0.08} validation macro F1: 0.54642


{'name': 't4_lgbm_15_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:36:21+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.5464207801524182,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_lgbm_15_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_lgbm_15_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [32]:
tune_family("lgbm_15", "engineered")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 100, 'min_child_samples': 60,...",0.69856,0.00901,0.69076,0.71118,0.69373
1,"{'n_estimators': 100, 'min_child_samples': 20,...",0.69756,0.00563,0.69031,0.70404,0.69831
2,"{'n_estimators': 180, 'min_child_samples': 20,...",0.67886,0.00737,0.67086,0.68864,0.67708


Selected: {'n_estimators': 100, 'min_child_samples': 60, 'learning_rate': 0.08} validation macro F1: 0.74278


{'name': 't4_lgbm_15_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:40:12+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.7427753812589513,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_lgbm_15_tuned_features_r1.json',
 'model_path': 'task4/save/t4_lgbm_15_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

## Lgbm 31 search

In [33]:
tune_family("lgbm_31", "original")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 100, 'min_child_samples': 60,...",0.51550,0.00341,0.51256,0.51366,0.52028
1,"{'n_estimators': 100, 'min_child_samples': 20,...",0.50345,0.00085,0.50287,0.50465,0.50284
2,"{'n_estimators': 180, 'min_child_samples': 20,...",0.49971,0.00047,0.49955,0.50035,0.49924


Selected: {'n_estimators': 100, 'min_child_samples': 60, 'learning_rate': 0.08} validation macro F1: 0.58965


{'name': 't4_lgbm_31_tuned_pixels_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:41:58+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.5896534631548933,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_lgbm_31_tuned_pixels_r1.json',
 'model_path': 'task4/save/t4_lgbm_31_tuned_pixels_r1.joblib',
 'data_name': 'eda_original_v1'}

In [34]:
tune_family("lgbm_31", "engineered")


,params,mean_macro_F1,std_macro_F1,fold_1_macro_F1,fold_2_macro_F1,fold_3_macro_F1
0,"{'n_estimators': 100, 'min_child_samples': 60,...",0.70094,0.00588,0.69667,0.70926,0.69690
1,"{'n_estimators': 100, 'min_child_samples': 20,...",0.69995,0.00395,0.69442,0.70205,0.70338
2,"{'n_estimators': 180, 'min_child_samples': 20,...",0.68556,0.01027,0.68116,0.69975,0.67577


Selected: {'n_estimators': 100, 'min_child_samples': 60, 'learning_rate': 0.08} validation macro F1: 0.77164


{'name': 't4_lgbm_31_tuned_features_r1',
 'task': 'task4',
 'evaluated_at': '2026-11-06T06:48:08+00:00',
 'split': 'validation',
 'card': {'split': 'held-out fold 1/5 after development-only random search',
  'seed': 42,
  'search_rows': 9000},
 'metric': 'macro_F1',
 'score': 0.7716444859086071,
 'higher_is_better': True,
 'model_card_path': 'task4/save/t4_lgbm_31_tuned_features_r1.json',
 'model_path': 'task4/save/t4_lgbm_31_tuned_features_r1.joblib',
 'data_name': 'eda_1_v1'}

# Compare and diagnose

In [35]:
index = read_evaluation_index(4)
names = [n for n in index["evaluations"] if n.endswith("_r1")]
ranking = pd.DataFrame([{"model": n, "macro_F1": index["evaluations"][n]["score"]} for n in names]).sort_values("macro_F1", ascending=False)
display(ranking)
family = {"dummy": "Majority", "logreg": "Logistic", "svm": "Linear SVM (SGD)",
          "extratrees": "ExtraTrees", "hgb": "HistGradientBoosting"}
def family_name(name):
    parts = name.split("_")
    return family.get(parts[1], "LightGBM tuned" if parts[2] == "tuned" else "LightGBM " + parts[2] + " leaves")
paired = ranking.assign(family=ranking.model.map(family_name),
                        variant=ranking.model.map(lambda name: "tuned" if "_tuned_" in name else "baseline"),
                        feature_set=ranking.model.map(lambda name: "engineered" if "features" in name else "original"))
display(paired.pivot(index=["family", "variant"], columns="feature_set", values="macro_F1").round(5))
cv_rows = []
for dataset, tuned_name in [("original", "t4_lgbm_tuned_pixels_r1"),
                            ("engineered", "t4_lgbm_tuned_features_r1")]:
    tuned_card = json.loads((SAVE_DIR / f"{tuned_name}.json").read_text(encoding="utf-8"))["card"]
    best_row = max(tuned_card["cv_results"], key=lambda row: row["mean_macro_F1"])
    heldout = index["evaluations"][tuned_name]["score"]
    cv_rows.append({"features": dataset, "best_params": tuned_card["best_params"],
                    "CV_mean": best_row["mean_macro_F1"], "CV_std": best_row["std_macro_F1"],
                    "validation": heldout, "validation_minus_CV": heldout - best_row["mean_macro_F1"]})
cv_readout = pd.DataFrame(cv_rows).set_index("features")
print("LightGBM search: 3 development folds; validation was excluded from tuning")
display(cv_readout.round(5))
print("Engineered minus original CV macro F1:", round(cv_readout.loc["engineered", "CV_mean"] - cv_readout.loc["original", "CV_mean"], 5))

tuning_rows = []
for family in TUNE_SPECS:
    for dataset in (["original"] if family == "dummy" else ["original", "engineered"]):
        tuned_name = f"t4_{family}_tuned_{'pixels' if dataset == 'original' else 'features'}_r1"
        card = json.loads((SAVE_DIR / f"{tuned_name}.json").read_text(encoding="utf-8"))["card"]
        best_cv = max(card["cv_results"], key=lambda row: row["mean_macro_F1"])
        tuning_rows.append({"family": family, "features": dataset,
                            "CV_mean": best_cv["mean_macro_F1"], "CV_std": best_cv["std_macro_F1"],
                            "validation": index["evaluations"][tuned_name]["score"]})
print("All-family random search on the same 9,000-image development sample:")
display(pd.DataFrame(tuning_rows).sort_values("CV_mean", ascending=False).round(5))
winner_name = ranking.iloc[0]["model"]
winner_card = json.loads((SRC / "task4" / "save" / f"{winner_name}.json").read_text(encoding="utf-8"))
winner = joblib.load(SRC / winner_card["model_path"])
feature_set = "original" if winner_card["data_name"] == "eda_original_v1" else "engineered"
winner_pred = winner.predict(datasets[feature_set][features[feature_set]].iloc[valid_idx].astype("float32"))
print("Winner:", winner_name)
print(classification_report(y.iloc[valid_idx], winner_pred, labels=labels, zero_division=0))
display(pd.DataFrame(confusion_matrix(y.iloc[valid_idx], winner_pred, labels=labels), index=labels, columns=labels))

,model,macro_F1
27,t4_lgbm_31_tuned_features_r1,0.771644
7,t4_lgbm_31_features_r1,0.762675
14,t4_lgbm_tuned_features_r1,0.762675
23,t4_hgb_tuned_features_r1,0.752871
25,t4_lgbm_15_tuned_features_r1,0.742775
6,t4_lgbm_15_features_r1,0.729200
2,t4_logreg_features_r1,0.721277
17,t4_logreg_tuned_features_r1,0.714570
5,t4_hgb_features_r1,0.710466
3,t4_svm_features_r1,0.678589


feature_set                    engineered  original
family               variant                       
ExtraTrees           baseline     0.61219   0.52570
                     tuned        0.59999   0.49213
HistGradientBoosting baseline     0.71047   0.52485
                     tuned        0.75287   0.57489
LightGBM 15 leaves   baseline     0.72920   0.53547
                     tuned        0.74278   0.54642
LightGBM 31 leaves   baseline     0.76267   0.58930
                     tuned        0.77164   0.58965
LightGBM tuned       tuned        0.76267   0.59073
Linear SVM (SGD)     baseline     0.67859   0.33276
                     tuned        0.67655   0.35987
Logistic             baseline     0.72128   0.41192
                     tuned        0.71457   0.43290
Majority             baseline         NaN   0.11305
                     tuned            NaN   0.20404

LightGBM search: 3 development folds; validation was excluded from tuning


,best_params,CV_mean,CV_std,validation,validation_minus_CV
features,,,,,
original,"{'min_child_samples': 60, 'num_leaves': 31}",0.57794,0.00393,0.59073,0.01279
engineered,"{'min_child_samples': 20, 'num_leaves': 31}",0.75737,0.00244,0.76267,0.00531


Engineered minus original CV macro F1: 0.17942
All-family random search on the same 9,000-image development sample:


,family,features,CV_mean,CV_std,validation
12,lgbm_31,engineered,0.70094,0.00588,0.77164
10,lgbm_15,engineered,0.69856,0.00901,0.74278
8,hgb,engineered,0.69177,0.00480,0.75287
2,logreg,engineered,0.64594,0.00757,0.71457
4,svm,engineered,0.62685,0.00221,0.67655
11,lgbm_31,original,0.51550,0.00341,0.58965
6,extratrees,engineered,0.51385,0.00054,0.59999
7,hgb,original,0.50368,0.00449,0.57489
9,lgbm_15,original,0.49129,0.00301,0.54642
5,extratrees,original,0.41183,0.00237,0.49213


Winner: t4_lgbm_31_tuned_features_r1
              precision    recall  f1-score   support

       7-11k       0.81      0.93      0.87      2486
       fiveN       0.83      0.74      0.78       882
   freefried       0.79      0.77      0.78      1119
        pube       0.78      0.72      0.75      1185
     roblock       0.82      0.58      0.68       638

    accuracy                           0.80      6310
   macro avg       0.80      0.75      0.77      6310
weighted avg       0.80      0.80      0.80      6310



,7-11k,fiveN,freefried,pube,roblock
7-11k,2318,33,48,64,23
fiveN,110,654,43,52,23
freefried,143,32,862,75,7
pube,160,30,109,859,27
roblock,138,42,30,57,371


# Round 1 readout

The all-family table compares searches on identical development rows and grouped folds; each chosen model is then refitted on all development images. CV spread and the held-out gap show how stable each candidate looks. This round remains exploratory because earlier baseline results were inspected on the same final validation fold. The majority classifier is a reference, not a serious tuning target.